<a href="https://colab.research.google.com/github/naubergois/curso-segmentacao-colab/blob/main/exercicios/aula2-exercicio3-quadros-com-pessoa.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 2, exercício 3: Quadros com pessoa

Em quantos dos primeiros 40 quadros processados aparece pelo menos uma pessoa?

O exercício é da aula [2. Segmentação em vídeo](https://colab.research.google.com/github/naubergois/curso-segmentacao-colab/blob/main/notebooks/02-segmentacao-em-video.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara os dados, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [ ]:
!pip install -q ultralytics

In [ ]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mascaras_originais(resultado):
    """Máscaras no tamanho da foto, como array booleano (n, altura, largura).

    A rede responde numa grade menor que a imagem. Aqui a máscara é esticada
    de volta ao tamanho original, para a área bater com os pixels da foto.
    """
    import cv2
    import numpy as np

    altura, largura = resultado.orig_shape
    if resultado.masks is None or len(resultado.masks) == 0:
        return np.zeros((0, altura, largura), dtype=bool)
    saida = []
    for mascara in resultado.masks.data.cpu().numpy():
        mascara = cv2.resize(mascara, (largura, altura), interpolation=cv2.INTER_NEAREST)
        saida.append(mascara > 0.5)
    return np.stack(saida)


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

## Preparação

A preparação baixa o vídeo da obra e segmenta os primeiros 80 quadros. Ficam prontos `pixels`, `classes_por_quadro` e `mascaras_por_quadro`.

In [ ]:
from collections import Counter

import cv2
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
video = baixar(
    "https://raw.githubusercontent.com/naubergois/curso-segmentacao-colab/main/midia/obra.mp4",
    "obra.mp4",
)

MAX_QUADROS = 80
cap = cv2.VideoCapture(video)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
pixels = Counter()
classes_por_quadro = []
mascaras_por_quadro = []
quadros = 0
while quadros < MAX_QUADROS:
    ok, frame = cap.read()
    if not ok:
        break
    r = model(frame, verbose=False)[0]
    nomes = []
    if r.masks is not None:
        for c, m in zip(r.boxes.cls.int().tolist(), mascaras_originais(r)):
            nome = model.names[c]
            nomes.append(nome)
            pixels[nome] += int(m.sum())
    classes_por_quadro.append(nomes)
    mascaras_por_quadro.append(len(nomes))
    quadros += 1
cap.release()

print(f"{largura}x{altura} a {fps} FPS, {quadros} quadros")
print(pixels.most_common(3))

In [ ]:
# @title Correção automática (rode, não precisa ler)
from functools import lru_cache
from pathlib import Path

_cap = cv2.VideoCapture(video)
_total = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
_fps_arquivo = _cap.get(cv2.CAP_PROP_FPS)
_cap.release()


@lru_cache(None)
def _pixels_pessoa_pulando():
    cap = cv2.VideoCapture(video)
    total = 0
    for i in range(40):
        ok, frame = cap.read()
        if not ok:
            break
        if i % 5:
            continue
        r = model(frame, verbose=False)[0]
        if r.masks is None:
            continue
        for c, m in zip(r.boxes.cls.int().tolist(), mascaras_originais(r)):
            if model.names[c] == "person":
                total += int(m.sum())
    cap.release()
    return total


VERIFICAR = {
    1: lambda r: abs(float(r) - _total / _fps_arquivo) < 0.05,
    2: lambda r: r in {nome for nome, n in pixels.items() if n == max(pixels.values())},
    3: lambda r: int(r) == sum("person" in nomes for nomes in classes_por_quadro[:40]),
    4: lambda r: int(r) == _pixels_pessoa_pulando(),
    5: lambda r: abs(float(r) - (sum(mascaras_por_quadro) / len(mascaras_por_quadro))) < 1e-3,
    6: lambda r: Path(str(r)).name == "obra_pessoas.mp4" and Path(str(r)).stat().st_size > 1000,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Em quantos dos primeiros 40 quadros processados aparece pelo menos uma pessoa?

*Dica:* classes_por_quadro[i] é a lista de nomes daquele quadro.

Troque o `None` pela sua resposta e rode a célula.

In [ ]:
resposta = None

conferir(3, resposta)

## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [ ]:
resposta = sum("person" in nomes for nomes in classes_por_quadro[:40])

conferir(3, resposta)

## O que fica

Próximo: [Aula 2, exercício 4: Pular quadros](https://colab.research.google.com/github/naubergois/curso-segmentacao-colab/blob/main/exercicios/aula2-exercicio4-pular-quadros.ipynb).